In [1]:
import torch
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import os

# --- CONFIGURATION ---
# These parameters MUST match the ones used in the training/generation scripts.
SEQ_LEN = 12      # Input history length
PRED_LEN = 12     # Output forecast horizon
DATA_PATH = '../../../ICL4DT/data/time_series_datasets/ETTm2.csv'
HTI_DATA_DIR = 'hti_data_long'

# The quantiles of the expert models you want to visualize
QUANTILES_TO_LOAD = [0.01, 0.1, 0.25, 0.5, 0.75,0.9, 0.99]

print("Loading original dataset...")
df = pd.read_csv(DATA_PATH)
data = df['OT'].values.astype(float)

print(f"Full dataset shape: {data.shape}")

# Recreate the exact train/val/test split to fit the scaler correctly
train_split_idx = int(len(data) * 0.7)
val_split_idx = int(len(data) * 0.98)

# Isolate the original, unscaled test data for ground truth comparison
original_test_data = data[val_split_idx:]

# Fit the scaler ONLY on the training data to prevent data leakage
print("Fitting MinMaxScaler on the training data portion...")
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(data[:train_split_idx].reshape(-1, 1))

hti_datasets = {}
all_forecasts_unscaled = {}

print("Loading HTI datasets and unscaling forecasts...")

for q in QUANTILES_TO_LOAD:
    # Construct filename (e.g., hti_data_q05.pt)
    filename = f"hti_data_q{str(q).replace('.', '')}.pt"
    
    try:
        # Load the entire [history, forecast] tensor
        hti_datasets[q] = torch.load(filename)
        data = hti_datasets[q]
        print(f" -> Loaded '{filename}' with shape: {hti_datasets[q].shape}")
        data_unscaled = scaler.inverse_transform(data)
        data_unscaled = torch.tensor(data_unscaled, dtype=torch.float32)
        all_forecasts_unscaled[q] = data_unscaled
        
    except FileNotFoundError:
        print(f" -> WARNING: Could not find file {filename}. Skipping.")

print("\nForecasts are now unscaled and ready for plotting.")


Loading original dataset...
Full dataset shape: (69680,)
Fitting MinMaxScaler on the training data portion...
Loading HTI datasets and unscaling forecasts...
 -> Loaded 'hti_data_q001.pt' with shape: torch.Size([1371, 24])
 -> Loaded 'hti_data_q01.pt' with shape: torch.Size([1371, 24])
 -> Loaded 'hti_data_q025.pt' with shape: torch.Size([1371, 24])
 -> Loaded 'hti_data_q05.pt' with shape: torch.Size([1371, 24])
 -> Loaded 'hti_data_q075.pt' with shape: torch.Size([1371, 24])
 -> Loaded 'hti_data_q09.pt' with shape: torch.Size([1371, 24])
 -> Loaded 'hti_data_q099.pt' with shape: torch.Size([1371, 24])

Forecasts are now unscaled and ready for plotting.


In [2]:
all_forecasts_unscaled

{0.01: tensor([[34.1700, 34.3895, 35.0485,  ..., 37.5624, 37.1149, 36.7910],
         [34.3895, 35.0485, 35.4885,  ..., 38.5167, 38.0795, 37.7563],
         [35.0485, 35.4885, 36.1475,  ..., 39.3353, 38.9040, 38.5929],
         ...,
         [47.0850, 47.0850, 47.0850,  ..., 43.7178, 43.1154, 43.0031],
         [47.0850, 47.0850, 47.0850,  ..., 43.4784, 42.8796, 42.7577],
         [47.0850, 47.0850, 47.0850,  ..., 43.1914, 42.5859, 42.4359]]),
 0.1: tensor([[34.1700, 34.3895, 35.0485,  ..., 40.5115, 40.1945, 40.0463],
         [34.3895, 35.0485, 35.4885,  ..., 42.5063, 42.1955, 42.0958],
         [35.0485, 35.4885, 36.1475,  ..., 43.3503, 43.0533, 42.9656],
         ...,
         [47.0850, 47.0850, 47.0850,  ..., 44.9479, 44.9051, 44.5218],
         [47.0850, 47.0850, 47.0850,  ..., 44.6607, 44.5947, 44.2068],
         [47.0850, 47.0850, 47.0850,  ..., 44.3946, 44.2931, 43.9033]]),
 0.25: tensor([[34.1700, 34.3895, 35.0485,  ..., 43.4155, 43.0947, 43.1299],
         [34.3895, 35.0485, 

In [3]:
combined = torch.stack([all_forecasts_unscaled[q] for q in QUANTILES_TO_LOAD], dim=0)

torch.save(combined, 'hti_data_combined.pt')

In [4]:
combined.shape

torch.Size([7, 1371, 24])

In [5]:
combined

tensor([[[34.1700, 34.3895, 35.0485,  ..., 37.5624, 37.1149, 36.7910],
         [34.3895, 35.0485, 35.4885,  ..., 38.5167, 38.0795, 37.7563],
         [35.0485, 35.4885, 36.1475,  ..., 39.3353, 38.9040, 38.5929],
         ...,
         [47.0850, 47.0850, 47.0850,  ..., 43.7178, 43.1154, 43.0031],
         [47.0850, 47.0850, 47.0850,  ..., 43.4784, 42.8796, 42.7577],
         [47.0850, 47.0850, 47.0850,  ..., 43.1914, 42.5859, 42.4359]],

        [[34.1700, 34.3895, 35.0485,  ..., 40.5115, 40.1945, 40.0463],
         [34.3895, 35.0485, 35.4885,  ..., 42.5063, 42.1955, 42.0958],
         [35.0485, 35.4885, 36.1475,  ..., 43.3503, 43.0533, 42.9656],
         ...,
         [47.0850, 47.0850, 47.0850,  ..., 44.9479, 44.9051, 44.5218],
         [47.0850, 47.0850, 47.0850,  ..., 44.6607, 44.5947, 44.2068],
         [47.0850, 47.0850, 47.0850,  ..., 44.3946, 44.2931, 43.9033]],

        [[34.1700, 34.3895, 35.0485,  ..., 43.4155, 43.0947, 43.1299],
         [34.3895, 35.0485, 35.4885,  ..., 45

In [6]:
combined_min = combined.min().item()
combined_max = combined.max().item()
print(f"Min value in combined: {combined_min}")
print(f"Max value in combined: {combined_max}")

Min value in combined: 21.102188110351562
Max value in combined: 59.46559143066406
